# Практикум 4. Разбиение выборки, baseline и первые метрики качества

В этом практикуме рассматривается базовая схема оценки регрессионной модели. До перехода к более сложным алгоритмам важно показать, как данные разделяются на части, зачем нужен baseline и как читать метрики качества без потери инженерного смысла.

## Почему без baseline нельзя двигаться дальше

Сложная модель сама по себе не является признаком качественного решения. Если не сравнить ее с простейшим ориентиром, нельзя понять, дает ли модель реальное улучшение или лишь создает впечатление вычислительной сложности.

## Что будет сделано в практикуме

- будет открыт тематический регрессионный набор;
- будут выделены признаки и целевая переменная;
- будет выполнено разбиение на обучающую и тестовую выборки;
- будет построен baseline-прогноз и линейная регрессия;
- будут сопоставлены `MAE`, `RMSE` и `R²` на одной визуальной схеме.

## Данные и инструменты

| Компонент | Назначение |
| --- | --- |
| `combined_cycle_power_plant.csv` | тематический набор для регрессии |
| `DummyRegressor` | baseline-ориентир |
| `LinearRegression` | первая содержательная модель |
| `sklearn.metrics` | расчет метрик качества |
| `matplotlib` / `seaborn` | графическое сравнение результатов |

## Ход работы

### Шаг 1. Открыть набор и выделить постановку задачи
Сначала зафиксируем, что именно прогнозируется и какие признаки для этого доступны.

### Шаг 2. Разделить данные на обучающую и тестовую части
Это нужно сделать до построения моделей, чтобы итоговая оценка оставалась честной.

### Шаг 3. Сравнить baseline и линейную регрессию
На этом этапе станет видно, действительно ли модель извлекает сигнал из признаков.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split

from src.display_utils import display_callout, display_frame, display_stage_note
from src.plot_utils import COURSE_COLORS, apply_course_plot_theme, format_axis
from src.project_paths import sample_data_dir

apply_course_plot_theme()

DATA_PATH = sample_data_dir() / "regression" / "combined_cycle_power_plant.csv"
df = pd.read_csv(DATA_PATH)

display_stage_note(
    "Шаг 1. Структура регрессионной задачи",
    "Откроем набор и зафиксируем состав признаков и целевую переменную.",
)

summary = pd.DataFrame(
    [
        {
            "Строк": len(df),
            "Признаков": 4,
            "Целевая переменная": "PE",
            "Пропусков": int(df.isna().sum().sum()),
        }
    ]
)
display_frame(summary, decimals=0)
display_frame(df.head(6))

## Почему этот набор удобен для первой регрессионной постановки

Здесь есть немного признаков, они интерпретируемы физически, а целевая величина явно задана. Благодаря этому внимание можно сосредоточить не на сложной очистке, а на правильной логике оценки качества модели.

In [ ]:
display_stage_note(
    "Шаг 2. Разбиение выборки",
    "Отделим тестовую часть до обучения моделей, чтобы итоговые метрики отражали качество на новых данных.",
)

X = df[["AT", "V", "AP", "RH"]]
y = df["PE"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

split_table = pd.DataFrame(
    [
        {"Подвыборка": "train", "Строк": len(X_train), "Доля": len(X_train) / len(df)},
        {"Подвыборка": "test", "Строк": len(X_test), "Доля": len(X_test) / len(df)},
    ]
)
display_frame(split_table)

## Что важно в схеме разбиения

Тестовая выборка не должна участвовать в обучении и настройке модели. Иначе итоговая оценка превращается в самооценку на уже знакомых наблюдениях и перестает быть надежным ориентиром для практики.

In [ ]:
display_stage_note(
    "Шаг 3. Baseline и линейная регрессия",
    "Построим простой ориентир и сравним его с первой содержательной моделью на одних и тех же тестовых данных.",
)

baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

linear_model = LinearRegression()
linear_model.fit(X_train, y_train)
linear_pred = linear_model.predict(X_test)

comparison = pd.DataFrame(
    [
        {
            "Модель": "Baseline (среднее)",
            "MAE": mean_absolute_error(y_test, baseline_pred),
            "RMSE": root_mean_squared_error(y_test, baseline_pred),
            "R2": r2_score(y_test, baseline_pred),
        },
        {
            "Модель": "Линейная регрессия",
            "MAE": mean_absolute_error(y_test, linear_pred),
            "RMSE": root_mean_squared_error(y_test, linear_pred),
            "R2": r2_score(y_test, linear_pred),
        },
    ]
)
display_frame(comparison)

In [ ]:
metrics_long = comparison.melt(id_vars="Модель", var_name="Метрика", value_name="Значение")
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

sns.barplot(
    data=metrics_long[metrics_long["Метрика"].isin(["MAE", "RMSE"])],
    x="Метрика",
    y="Значение",
    hue="Модель",
    palette=[COURSE_COLORS["accent_light"], COURSE_COLORS["accent"]],
    ax=axes[0],
)
format_axis(
    axes[0],
    title="Сравнение ошибок прогноза",
    subtitle="Меньшие значения MAE и RMSE соответствуют лучшему качеству",
    xlabel="Метрика",
    ylabel="Значение",
)

axes[1].scatter(y_test, linear_pred, color=COURSE_COLORS["accent"], alpha=0.55)
line_min = min(y_test.min(), linear_pred.min())
line_max = max(y_test.max(), linear_pred.max())
axes[1].plot([line_min, line_max], [line_min, line_max], linestyle="--", color=COURSE_COLORS["highlight"])
format_axis(
    axes[1],
    title="Фактические и прогнозные значения",
    subtitle="Чем ближе точки к диагонали, тем точнее прогноз линейной модели",
    xlabel="Фактическая мощность, МВт",
    ylabel="Прогнозная мощность, МВт",
)
plt.tight_layout()
plt.show()

## Как читать полученные результаты

Сводная таблица метрик показывает количественное превосходство линейной модели над baseline. Бар-чарт делает это сравнение наглядным, а диаграмма рассеяния позволяет увидеть, насколько прогнозные значения приближены к фактическим. Такой набор визуализаций помогает не просто назвать числа, а понять характер качества модели.

In [ ]:
display_callout(
    "Методический итог",
    "Схема «разбиение -> baseline -> содержательная модель -> интерпретация метрик» должна предшествовать любому усложнению модели. Иначе сравнение результатов теряет аналитическую строгость.",
    tone="success",
)

## Итоговый вывод

Даже простая линейная регрессия имеет смысл только на фоне корректного baseline и честного тестирования. Именно такая последовательность делает метрики качества содержательным аргументом, а не формальным приложением к коду.

## Вопросы для самостоятельной работы

1. Почему baseline со средним значением остается полезным даже для сильных моделей?
2. В каких случаях `MAE` удобнее интерпретировать, чем `RMSE`?
3. Как изменилась бы схема оценки, если бы данные представляли временной ряд, а не обычную табличную выборку?

## Источники

1. [Глава 4 пособия](https://github.com/Nephalem72/power-data-analysis-handbook/blob/main/docs/chapters/04-splitting-metrics-and-baselines.md)
2. [Паспорт датасетов](https://github.com/Nephalem72/power-data-analysis-handbook/blob/main/docs/datasets.md)
3. [Combined Cycle Power Plant, UCI](https://archive.ics.uci.edu/dataset/294/combined+cycle+power+plant)